In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS

from scipy import sparse
from xarray import DataArray
from scipy.sparse.linalg import eigsh
import numpy as np

import pyvista as pv
import pyansys

from pyFBS.utility import *


In [3]:
receiver = "../data/AM_automotive_testbench/STL/receiver.stl"

In [4]:
stl = r"../data/lab_testbench/STL/AB.stl"
xlsx = r"../data/lab_testbench/Measurements/AM_Measurements.xlsx"

full_file = r'..\data\lab_testbench\FEM\AB\file.full'
ress_file = r'..\data\lab_testbench\FEM\AB\file.rst'

In [5]:
MK = pyFBS.MK_model(ress_file,full_file,no_modes = 100,recalculate = False)

In [6]:
view3D = pyFBS.display.view3D(show_origin= True)

In [7]:
#view3D = pyFBS.display.view3D()
view3D.add_stl(stl,name = "engine_mount",color = "#8FB1CC",opacity = 0.8)

In [8]:
view3D.plot.add_mesh(MK.mesh, scalars = np.ones(MK.mesh.points.shape[0]),show_scalar_bar = False,name = "mesh",cmap = "coolwarm",show_edges = True,render_points_as_spheres  = True,style = "surface")

(vtkRenderingOpenGL2Python.vtkOpenGLActor)000001ADF9C2A528

In [16]:
select_mode = 2
_modeshape = MK.get_modeshape(select_mode)

mode_dict = dict_animation(_modeshape,MK.pts,MK.mesh,"modeshape")
view3D.add_modeshape(mode_dict,run_animation = True)

[autoreload of pyFBS.display failed: Traceback (most recent call last):
  File "C:\Users\tomaz.bregar\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\extensions\autoreload.py", line 245, in check
    superreload(m, reload, self.old_objects)
  File "C:\Users\tomaz.bregar\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\extensions\autoreload.py", line 450, in superreload
    update_generic(old_obj, new_obj)
  File "C:\Users\tomaz.bregar\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\extensions\autoreload.py", line 387, in update_generic
    update(a, b)
  File "C:\Users\tomaz.bregar\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\extensions\autoreload.py", line 357, in update_class
    update_instances(old, new)
  File "C:\Users\tomaz.bregar\AppData\Local\Continuum\anaconda3\lib\site-packages\IPython\extensions\autoreload.py", line 312, in update_instances
    update_instances(old, new, obj.__dict__, visited)
  File "C:\Users\tomaz.bregar\A

AttributeError: 'NoneType' object has no attribute 'GetNumberOfPoints'

In [13]:
df_imp = pd.read_excel(xlsx, sheet_name='Impacts_AB')
view3D.show_imp(df_imp,overwrite = True)
#view3D.label_imp(df_imp)
#df_imp

In [11]:
df_chn = pd.read_excel(xlsx, sheet_name='Channels_AB')
view3D.show_chn(df_chn)
#df_chn

In [15]:
df_chn_up = MK.update_locations_df(df_chn)
df_imp_up = MK.update_locations_df(df_imp)

view3D.show_imp(df_imp_up, color = "k",overwrite = False)


In [ ]:
MK.FRF_synth(df_chn,df_imp,modal_damping = 0.003,frf_type = "accelerance")

In [ ]:
freq, Y_AB_exp = np.load(r"../data/lab_testbench/Measurements/Y_AB.p",allow_pickle = True)

In [ ]:
plt.figure(figsize = (12,8))

s1 = 5
s2 = 0

display(df_chn.iloc[[s1]])
display(df_imp.iloc[[s2]])

plt.subplot(211)
plt.semilogy(MK.freq,np.abs(MK.FRF[:,s1,s2]))
plt.semilogy(freq,np.abs(Y_AB_exp[s1,s2]))


plt.subplot(413)
plt.plot(MK.freq,np.angle(MK.FRF[:,s1,s2]))
plt.plot(freq,np.angle(Y_AB_exp[s1,s2]))



In [ ]:
#df_chn = pd.read_excel(xlsx, sheet_name='Channels_AB')
#df_imp = pd.read_excel(xlsx, sheet_name='Impacts_AB')

df_chn = df_chn_up
df_imp = df_imp_up

df_vp = pd.read_excel(xlsx, sheet_name='VP_Channels')
df_vpref = pd.read_excel(xlsx, sheet_name='VP_RefChannels')

vpt = pyFBS.VPT(df_chn,df_imp,df_vp,df_vpref)

In [ ]:
frf_exp = np.transpose(Y_AB_exp,(2,0,1))

vpt.apply_VPT(MK.FRF,MK.FRF)
vpt.consistency([1],[1])

In [ ]:
vpt.vptData.shape

In [ ]:
plt.semilogy(np.abs(vpt.u[8,:]))
plt.semilogy(np.abs(vpt.u_f[8,:]))

In [ ]:
plt.bar(range(9),vpt.specific_sensor)

In [ ]:
plt.bar(range(9),vpt.specific_impact)

In [ ]:
Y_SEMM_coh = np.zeros((24,24))

for i in range(24):
    for j in range(24):
        Y_SEMM_coh[i,j] = coh_frf(vpt.vptData[:,i,j], vpt.vptData[:,j,i])

plt.imshow(Y_SEMM_coh)